# Edge TTS — Arabic number pronunciation mapping test

Chosen: `edge_tts_numbers_mapping_v1/chosen/` · Comparisons + cuts: `.../comparison/`

Voice priority: `ar-EG-SalmaNeural`, then `ar-EG-ShakirNeural`.

TTS input is **only** the variant text.



## 1. Dependencies


In [1]:
import subprocess
import sys

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
        print(f"[deps OK] {pkg}")
        return True
    except ImportError:
        print(f"[deps] installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)
        print(f"[deps OK] {pkg} (installed)")
        return True

_ensure("edge-tts", "edge_tts")
_ensure("nest_asyncio")
_ensure("IPython", "IPython")
print("Edge TTS dependency bootstrap done.")


[deps OK] edge-tts
[deps OK] nest_asyncio
[deps OK] IPython
Edge TTS dependency bootstrap done.


## 2. Paths and output folders


In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
AUDIO_ROOT = BASE_DIR / "generated_audio"
MAPPING_ROOT = AUDIO_ROOT / "edge_tts_numbers_mapping_v1"
DIR_CHOSEN = MAPPING_ROOT / "chosen"
DIR_COMPARISON = MAPPING_ROOT / "comparison"
DIR_COMPARISON_CUT = DIR_COMPARISON / "cut"
DIR_NUMBERS = AUDIO_ROOT / "edge_tts_numbers"
DIR_CHOSEN_V2_FALLBACK = AUDIO_ROOT / "edge_tts_mapping_v2" / "chosen"

for d in (DIR_CHOSEN, DIR_COMPARISON, DIR_COMPARISON_CUT, DIR_NUMBERS):
    d.mkdir(parents=True, exist_ok=True)

print("cwd:", BASE_DIR)
print("chosen:", DIR_CHOSEN)
print("comparison:", DIR_COMPARISON)
print("comparison/cut:", DIR_COMPARISON_CUT)
print("numbers source:", DIR_NUMBERS)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff
chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\chosen
comparison: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\comparison
comparison/cut: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\comparison\cut
numbers source: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers


## 3. Chosen moves and round-2 comparison variants


In [3]:
# v1/chosen: comparison winner for number 0
CHOSEN_V1_STEMS = ["num_00_v02"]

CHOSEN_COMPARISON_PROMOTE = ["num_10_asharah_sukoon"]

CHOSEN_FROM_NUMBERS = [1, 3, 4, 5, 6, 7, 8, 9]

CUT_SOURCE_STEM = "num_02_atneen_bang"
CUT_TARGET_DURATION_SEC = 0.59
CUT_OUTPUT_NAME = "num_02_atneen_bang_cut_0_59.mp3"

ROUND2_COMPARISON: dict[int, list[tuple[str, str]]] = {
    2: [
        ("num_02_itneen_sukoon", "اِتْنِينْ."),
        ("num_02_atneen_sukoon", "اتنينْ."),
        ("num_02_etneen_sukoon_ar", "إتنينْ."),
        ("num_02_itneen_plain", "اِتنين."),
        ("num_02_atneen_plain", "اتنين."),
        ("num_02_etneen_lat", "etneen."),
        ("num_02_itneen_lat", "itneen."),
        ("num_02_etnein_lat", "etnein."),
        ("num_02_atneen_bang", "اتنين!"),
    ],
    10: [
        ("num_10_ashara_ta_sukoon", "عشرةْ."),
        ("num_10_ashr_ah_sukoon", "عَشْرَهْ."),
        ("num_10_asharah_ta", "عَشَرَة."),
        ("num_10_ashara_plain", "عَشرة."),
        ("num_10_ashara_lat", "ashara."),
        ("num_10_asharah_lat", "asharah."),
        ("num_10_ashara_ipa", "ʿashara."),
        ("num_10_ashara_bang", "عشرة!"),
    ],
}

ROUND1_REMOVE_PREFIXES = [0, 2, 10]


def with_period(text: str) -> str:
    t = text.strip()
    if t.endswith((".", "!", "?")):
        return t
    return t + "."


def tts_text_for_variant(variant: str) -> str:
    return with_period(variant)


print("Promote to chosen:", CHOSEN_COMPARISON_PROMOTE)
print("From edge_tts_numbers:", CHOSEN_FROM_NUMBERS)
print("Cut:", CUT_SOURCE_STEM, "->", CUT_OUTPUT_NAME)


Promote to chosen: ['num_10_asharah_sukoon']
From edge_tts_numbers: [1, 3, 4, 5, 6, 7, 8, 9]
Cut: num_02_atneen_bang -> num_02_atneen_bang_cut_0_6.mp3


## 4. Edge TTS voice selection


In [4]:
import asyncio

import edge_tts
import nest_asyncio

nest_asyncio.apply()

PREFERRED_VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]
EDGE_VOICE = None
EDGE_TTS_LOADED = False
_ALL_VOICES = None

async def _list_voices():
    global _ALL_VOICES
    if _ALL_VOICES is None:
        _ALL_VOICES = await edge_tts.list_voices()
    return _ALL_VOICES

async def select_edge_voice() -> str:
    voices = await _list_voices()
    by_short = {v["ShortName"]: v for v in voices}
    for pref in PREFERRED_VOICES:
        if pref in by_short:
            return pref
    ar_names = sorted(
        v["ShortName"]
        for v in voices
        if str(v.get("Locale", "")).lower().startswith("ar")
    )
    print("Preferred voices unavailable. Available Arabic voices:")
    for name in ar_names:
        print(" ", name)
    eg = [
        v["ShortName"]
        for v in voices
        if "EG" in v.get("Locale", "") or "-EG-" in v.get("ShortName", "")
    ]
    if eg:
        return eg[0]
    if ar_names:
        return ar_names[0]
    raise RuntimeError("No Arabic Edge TTS voice found")

EDGE_VOICE = asyncio.get_event_loop().run_until_complete(select_edge_voice())
EDGE_TTS_LOADED = True
print(f"Selected Edge TTS voice: {EDGE_VOICE}")


Selected Edge TTS voice: ar-EG-SalmaNeural


## 5. Edge TTS synthesis (MP3 only)


In [5]:
import shutil
import time

from IPython.display import Audio as IPA, display

COMPARISON_LOG: list[dict] = []
PLAYBACK_SHOWN = False

async def synth_edge_mp3(text: str, mp3_path: Path) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    comm = edge_tts.Communicate(text, EDGE_VOICE)
    await comm.save(str(mp3_path))
    return time.perf_counter() - t0

async def generate_number_mp3(
    number: int,
    variant: str,
    mp3_path: Path,
) -> dict | None:
    tts_text = tts_text_for_variant(variant)
    try:
        infer_s = await synth_edge_mp3(tts_text, mp3_path)
    except Exception as exc:
        if mp3_path.is_file() and mp3_path.stat().st_size == 0:
            mp3_path.unlink()
        print(f"number: {number}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()}")
        print(f"FAILED (no audio): {exc}")
        print("---")
        return None
    if not mp3_path.is_file():
        print(f"number: {number}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()}")
        print("FAILED: file missing")
        print("---")
        return None
    entry = {
        "number": number,
        "variant": variant,
        "tts_text": tts_text,
        "mp3_path": mp3_path.resolve(),
        "infer_s": infer_s,
        "bytes": mp3_path.stat().st_size,
    }
    print(f"number: {number}")
    print(f"variant text sent to Edge TTS: {tts_text}")
    print(f"output MP3 path: {entry['mp3_path']}")
    print("---")
    return entry

def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


## 6. Move files to chosen (v1)

- `num_10_asharah_sukoon` from comparison → chosen (removed from comparison)
- `num_01`, `num_03`–`num_09` from `edge_tts_numbers/` → chosen


In [6]:
import re
import shutil

MOVED_TO_CHOSEN: list[Path] = []
REMOVED_FROM_OLD: list[Path] = []

print("=== Promote comparison winner -> chosen ===\n")

for stem in CHOSEN_COMPARISON_PROMOTE:
    src = DIR_COMPARISON / f"{stem}.mp3"
    if not src.is_file():
        print(f"SKIP missing in comparison: {src.name}")
        continue
    dest = DIR_CHOSEN / f"{stem}.mp3"
    if dest.is_file():
        dest.unlink()
    shutil.move(str(src), str(dest))
    MOVED_TO_CHOSEN.append(dest.resolve())
    print("moved to chosen:", dest.name)
    print("  output path:", dest.resolve())
    print("  removed from comparison:", src.name, "(moved)")

print("\n=== Move num_01, num_03-09 -> chosen v1 ===\n")

for n in CHOSEN_FROM_NUMBERS:
    name = f"num_{n:02d}.mp3"
    dest = DIR_CHOSEN / name
    src = DIR_NUMBERS / name
    if not src.is_file():
        if dest.is_file():
            print("already in chosen (source gone):", name)
            MOVED_TO_CHOSEN.append(dest.resolve())
        else:
            print(f"SKIP missing: {name}")
        continue
    if dest.is_file() and dest.resolve() != src.resolve():
        dest.unlink()
    shutil.move(str(src), str(dest))
    REMOVED_FROM_OLD.append(src.resolve())
    MOVED_TO_CHOSEN.append(dest.resolve())
    print("moved to chosen:", name)
    print("  from:", src.parent)
    print("  output path:", dest.resolve())
    print("  removed from:", src.parent.name + "/")

print(f"\nFiles moved to chosen: {len(MOVED_TO_CHOSEN)}")
print(f"Files removed from old folders: {len(REMOVED_FROM_OLD)}")


=== Promote comparison winner -> chosen ===

moved to chosen: num_10_asharah_sukoon.mp3
  output path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\chosen\num_10_asharah_sukoon.mp3
  removed from comparison: num_10_asharah_sukoon.mp3 (moved)

=== Move num_01, num_03-09 -> chosen v1 ===

SKIP missing: num_01.mp3
SKIP missing: num_03.mp3
SKIP missing: num_04.mp3
SKIP missing: num_05.mp3
SKIP missing: num_06.mp3
SKIP missing: num_07.mp3
SKIP missing: num_08.mp3
SKIP missing: num_09.mp3

Files moved to chosen: 1
Files removed from old folders: 0


## 6b. Cut `num_02_atneen_bang` (~0.59 s)

Keeps the start of the pronunciation. Original is not overwritten.


In [7]:
import subprocess
import sys

def _ensure_pkg(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        return __import__(name, fromlist=[""])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return __import__(name, fromlist=[""])

_ensure_pkg("mutagen")
_ensure_pkg("pydub")
from mutagen.mp3 import MP3
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

DIR_COMPARISON_CUT.mkdir(parents=True, exist_ok=True)

ORIG_PATH = DIR_COMPARISON / f"{CUT_SOURCE_STEM}.mp3"
CUT_PATH = DIR_COMPARISON_CUT / CUT_OUTPUT_NAME

if not ORIG_PATH.is_file():
    raise FileNotFoundError(f"Missing original: {ORIG_PATH}")

audio = AudioSegment.from_mp3(ORIG_PATH)
orig_duration_sec = len(audio) / 1000.0

nonsilent = detect_nonsilent(
    audio, min_silence_len=50, silence_thresh=audio.dBFS - 14, seek_step=10
)
if nonsilent:
    start_ms = nonsilent[0][0]
    audio = audio[start_ms:]

target_ms = int(CUT_TARGET_DURATION_SEC * 1000)
cut_audio = audio[:target_ms]
cut_audio.export(str(CUT_PATH), format="mp3")
cut_duration_sec = len(cut_audio) / 1000.0

print("=== Cut audio ===")
print("original duration:", f"{orig_duration_sec:.3f}s")
print("cut duration:", f"{cut_duration_sec:.3f}s")
print("saved path:", CUT_PATH.resolve())

print("\n--- Playback: original ---")
display(IPA(filename=str(ORIG_PATH)))
print("--- Playback: cut ---")
display(IPA(filename=str(CUT_PATH)))


=== Cut audio ===
original duration: 1.728s
cut duration: 0.600s
saved path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\comparison\cut\num_02_atneen_bang_cut_0_6.mp3

--- Playback: original ---


--- Playback: cut ---


## 7. Generate round-2 comparison variants (2 and 10 only)

Removes legacy round-1 `num_02_v*` / `num_10_v*` / other round-1 leftovers for 0 and 2 from comparison first.


In [8]:
print("=== Round-2 comparison generation ===")
print(f"Voice: {EDGE_VOICE}\n")

# Remove old round-1 comparison files for numbers we are re-testing or cleaning
for src in sorted(DIR_COMPARISON.glob("num_*.mp3")):
    stem = src.stem
    m = re.match(r"num_(\d{2})_", stem)
    if not m:
        continue
    n = int(m.group(1))
    if stem == CUT_SOURCE_STEM:
        continue
    if n in ROUND1_REMOVE_PREFIXES or n in ROUND2_COMPARISON:
        src.unlink()
        print("removed from comparison:", src.name)

COMPARISON_LOG.clear()
ok = 0
expected = sum(len(v) for v in ROUND2_COMPARISON.values())

for number in sorted(ROUND2_COMPARISON):
    for stem, variant in ROUND2_COMPARISON[number]:
        mp3_path = DIR_COMPARISON / f"{stem}.mp3"
        entry = run_async(generate_number_mp3(number, variant, mp3_path))
        if entry:
            COMPARISON_LOG.append(entry)
            ok += 1

print(f"\nGenerated {ok} / {expected} round-2 comparison MP3s")


=== Round-2 comparison generation ===
Voice: ar-EG-SalmaNeural

removed from comparison: num_02_atneen_bang.mp3
removed from comparison: num_02_atneen_plain.mp3
removed from comparison: num_02_atneen_sukoon.mp3
removed from comparison: num_02_etneen_lat.mp3
removed from comparison: num_02_etneen_sukoon_ar.mp3
removed from comparison: num_02_etnein_lat.mp3
removed from comparison: num_02_itneen_lat.mp3
removed from comparison: num_02_itneen_plain.mp3
removed from comparison: num_02_itneen_sukoon.mp3
removed from comparison: num_10_ashara_bang.mp3
removed from comparison: num_10_ashara_lat.mp3
removed from comparison: num_10_ashara_plain.mp3
removed from comparison: num_10_ashara_ta_sukoon.mp3
removed from comparison: num_10_asharah_lat.mp3
removed from comparison: num_10_asharah_ta.mp3
removed from comparison: num_10_ashr_ah_sukoon.mp3
number: 2
variant text sent to Edge TTS: اِتْنِينْ.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1

## 8. Playback — numbers 2 and 10


In [9]:
def _play_entry(entry: dict):
    print(f"  number: {entry['number']}")
    print(f"  variant text sent to Edge TTS: {entry['tts_text']}")
    print(f"  output MP3 path: {entry['mp3_path']}")
    display(IPA(filename=str(entry["mp3_path"])))
    print()

by_number: dict[int, list[dict]] = {}
for e in COMPARISON_LOG:
    by_number.setdefault(e["number"], []).append(e)

print("=== Playback — number 2 variants ===")
for entry in by_number.get(2, []):
    _play_entry(entry)

print("=== Playback — number 10 variants ===")
for entry in by_number.get(10, []):
    _play_entry(entry)

PLAYBACK_SHOWN = True


=== Playback — number 2 variants ===
=== Playback — number 10 variants ===


## 9. Verify folders


In [ ]:
chosen_files = sorted(DIR_CHOSEN.glob("*.mp3"))
comparison_files = sorted(DIR_COMPARISON.glob("num_*.mp3"))
cut_files = sorted(DIR_COMPARISON_CUT.glob("*.mp3"))

print("=== Verify folders ===")
print("Chosen:", DIR_CHOSEN.resolve())
print("Chosen MP3 count:", len(chosen_files))
for p in chosen_files:
    print(f"  {p.name}")

print("\nComparison (top-level num_*):", len(comparison_files))
for p in comparison_files:
    print(f"  {p.name}")

print("\nComparison/cut:", DIR_COMPARISON_CUT.resolve())
for p in cut_files:
    print(f"  {p.name}")

chosen_stems = {p.stem for p in chosen_files}
checks = [
    ("num_00_v02 in chosen", "num_00_v02" in chosen_stems),
    ("num_10_asharah_sukoon in chosen", "num_10_asharah_sukoon" in chosen_stems),
    ("num_10_asharah_sukoon not in comparison", not (DIR_COMPARISON / "num_10_asharah_sukoon.mp3").is_file()),
    ("num_01 in chosen", "num_01" in chosen_stems),
    ("num_03-09 in chosen", all(f"num_{n:02d}" in chosen_stems for n in CHOSEN_FROM_NUMBERS)),
    ("cut file exists", (DIR_COMPARISON_CUT / CUT_OUTPUT_NAME).is_file()),
    ("original bang not overwritten", (DIR_COMPARISON / f"{CUT_SOURCE_STEM}.mp3").is_file()),
]
print("\nChecks:")
for label, ok in checks:
    print(f"  [{'OK' if ok else 'FAIL'}] {label}")


=== Verify folders ===
Chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\chosen
Chosen MP3 count: 2
  num_00_v02.mp3
  num_10_asharah_sukoon.mp3

Comparison (top-level num_*): 0

Comparison/cut: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\comparison\cut
  num_02_atneen_bang_cut_0_6.mp3

Checks:
  [OK] num_00_v02 in chosen
  [OK] num_10_asharah_sukoon in chosen
  [OK] num_10_asharah_sukoon not in comparison
  [FAIL] num_01 in chosen
  [FAIL] num_03-09 in chosen
  [OK] cut file exists
  [FAIL] original bang not overwritten
